# Day 44 — Practice Questions

## Hierarchical clustering · DBSCAN

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Building the tree | 1 – 3 |
| **B** | Linkage | 4 – 6 |
| **C** | DBSCAN | 7 – 10 |
| **D** | Choosing | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> Replace every `...` with your own code. Where a question asks you to *explain*, write
> two or three sentences in the markdown cell — the explanation is the point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import load_iris, load_wine, make_moons, make_blobs, make_circles

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED = "#7B8AA1"
COLS = [TEAL, CORAL, VIOLET, BLUE, AMBER, "#C96FA0", "#5BA87F"]

def show(X, labels=None, title="", ax=None):
    ax = ax or plt.gca()
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], c="#9AA8BE", s=18, edgecolors="white", linewidths=.4)
    else:
        for u in sorted(set(labels)):
            m = labels == u
            if u == -1:
                ax.scatter(X[m, 0], X[m, 1], c=MUTED, marker="x", s=22, linewidths=1.2)
            else:
                ax.scatter(X[m, 0], X[m, 1], c=COLS[u % 7], s=18,
                           edgecolors="white", linewidths=.4)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([])

# ---- Part A works on six points you can check with a ruler ---------------
SIX = np.array([[0.0, 0.0], [0.5, 0.2], [4.0, 0.0], [4.4, 0.3], [2.0, 3.5], [2.3, 3.9]])

# ---- the rest uses the seeds-style wine data and some shapes -------------
WINE_X, WINE_Y = load_wine(return_X_y=True)
WINE_Z = StandardScaler().fit_transform(WINE_X)
print("six points:", SIX.shape, "   wine:", WINE_X.shape)

---
# Part A · Building the tree

## Question 1

Here are six points, laid out as three obvious pairs:

```
A (0.0, 0.0)   B (0.5, 0.2)      <- pair 1
C (4.0, 0.0)   D (4.4, 0.3)      <- pair 2
E (2.0, 3.5)   F (2.3, 3.9)      <- pair 3
```

Compute the full distance matrix yourself, print it, and say **which two points merge
first** and at what distance.

In [ ]:
D = np.sqrt(((SIX[:, None, :] - SIX[None, :, :]) ** 2).sum(axis=2))
print(pd.DataFrame(np.round(D, 3), index=list("ABCDEF"), columns=list("ABCDEF")))

off = D + np.eye(len(D)) * 999       # ignore the zero diagonal
i, j = ...
print("\nfirst merge:", "ABCDEF"[i], "and", "ABCDEF"[j], "at", round(off.min(), 4))

## Question 2

Build the full tree on the six points with `linkage(SIX, method='single')` and print the
merge matrix. Explain what each of the four columns means, and confirm the number of
merges is what you expect.

Then draw the dendrogram.

In [ ]:
L6 = linkage(SIX, method="single")
print(np.round(L6, 4))
print("\nmerges:", L6.shape[0], "  points:", len(SIX))

dendrogram(L6, labels=list("ABCDEF"))
plt.ylabel("distance merged at"); plt.show()

## Question 3

Now the wine data: 178 wines, 13 scaled chemical measurements, 3 real cultivars.

Build a `ward` tree, then print for `k = 2, 3, 4, 5` the agreement with the true
cultivars and the silhouette. Also print the merge heights of the last five merges and
the jump between each.

Then explain: **what would you have concluded about `k` if nobody had told you there were
three cultivars?**

In [ ]:
LW = linkage(WINE_Z, method="ward")

print(f"{'k':>3}{'agreement':>12}{'silhouette':>13}")
for k in [2, 3, 4, 5]:
    lab = fcluster(LW, k, criterion="maxclust")
    print(f"{k:>3}{...:>12.4f}{...:>13.4f}")

heights = LW[:, 2]
print(f"\n{'clusters left':>14}{'merge height':>15}{'jump':>10}")
for i in range(1, 6):
    print(f"{i:>14}{heights[-i]:>15.2f}{heights[-i] - heights[-i-1]:>10.2f}")

*Your answer:*

---
# Part B · Linkage

## Question 4

Run all four linkages on **three** different datasets at the right `k`:

- `make_blobs(300, centers=3, cluster_std=1.0, random_state=42)` — k = 3
- `make_moons(300, noise=0.06, random_state=0)` — k = 2
- `make_circles(300, factor=0.45, noise=0.05, random_state=0)` — k = 2

Print a table of agreements: four linkages down, three datasets across.

In [ ]:
SETS = [("blobs", *make_blobs(300, centers=3, cluster_std=1.0, random_state=42), 3),
        ("moons", *make_moons(300, noise=0.06, random_state=0), 2),
        ("circles", *make_circles(300, factor=0.45, noise=0.05, random_state=0), 2)]

out = {}
for lk in ["ward", "complete", "average", "single"]:
    out[lk] = {}
    for nm, D, y, k in SETS:
        Zs = StandardScaler().fit_transform(D)
        lab = ...
        out[lk][nm] = round(adjusted_rand_score(y, lab), 4)

print(pd.DataFrame(out).T)

## Question 5

Now find the case where single linkage is the **worst** choice.

Build two groups of different density — 120 tight points and 120 loose ones — and run all
four linkages at `k = 2`.

In [ ]:
r = np.random.default_rng(1)
XU = np.vstack([r.normal([0, 0], 0.28, (120, 2)),      # tight
                r.normal([3.8, 0], 1.10, (120, 2))])   # loose
YU = np.r_[np.zeros(120), np.ones(120)].astype(int)

for lk in ["ward", "complete", "average", "single"]:
    lab = ...
    print(f"  {lk:9} agreement {adjusted_rand_score(YU, lab):.4f}"
          f"   sizes {sorted(np.bincount(lab).tolist())}")

## Question 6

You have now seen single linkage score **1.000** on three datasets and **0.000** on a
fourth.

Explain in your own words what single linkage actually measures, why that makes it
brilliant on the moons and circles, and why the same property destroys it on the uneven
density data. Then say which linkage you would use by default, and why.

*Your answer:*

---
# Part C · DBSCAN

## Question 7

Work out the three kinds of point **yourself**, without using `DBSCAN`.

For the dataset below with `eps = 0.5` and `min_samples = 5`:

- a point is **core** if at least `min_samples` points (including itself) lie within `eps`
- a point is **border** if it is not core but lies within `eps` of some core point
- otherwise it is **noise**

Print the three counts, then check them against sklearn.

In [ ]:
rg = np.random.default_rng(9)
PT = np.vstack([rg.normal([0, 0], 0.5, (60, 2)),
                rg.normal([3.0, 2.0], 0.5, (55, 2)),
                rg.uniform([-2, -2], [5, 4], (15, 2))])
EPS, MS = 0.5, 5

D = np.sqrt(((PT[:, None, :] - PT[None, :, :]) ** 2).sum(axis=2))
is_core = ...
is_border = ...
is_noise = ...

print("core  ", is_core.sum())
print("border", is_border.sum())
print("noise ", is_noise.sum())

db = DBSCAN(eps=EPS, min_samples=MS).fit(PT)
sk_core = np.zeros(len(PT), bool); sk_core[db.core_sample_indices_] = True
print("\nmatches sklearn:", bool((sk_core == is_core).all()))

## Question 8

Break it with `eps`. On `make_moons(300, noise=0.06, random_state=0)` sweep `eps` over
`[0.05, 0.10, 0.15, 0.20, 0.30, 0.50]` with `min_samples=5`, and print the number of
clusters found, the number of noise points, and the agreement.

In [ ]:
XM, YM = make_moons(300, noise=0.06, random_state=0)

print(f"{'eps':>6}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XM)
    nc = ...
    print(f"{eps:>6}{nc:>10}{(d.labels_ == -1).sum():>8}{...:>12.4f}")

## Question 9

Use the **k-distance recipe** to choose `eps` without looking at the labels: measure each
point's distance to its 5th nearest neighbour, sort them, take the 90th percentile.

Apply it to the moons. Compare the agreement you get to the best value from Question 8,
and explain what the recipe is and is not good for.

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=5).fit(XM).kneighbors(XM)
kd = np.sort(dist[:, -1])
suggested = ...

plt.figure(figsize=(6, 3))
plt.plot(kd); plt.axhline(suggested, color=CORAL, ls="--")
plt.xlabel("points, sorted"); plt.ylabel("distance to 5th neighbour"); plt.show()

print("recipe suggests eps =", round(suggested, 4))
print("agreement there     =", ...)
print("best from Q8 (0.20) =", round(adjusted_rand_score(
    YM, DBSCAN(eps=0.20, min_samples=5).fit(XM).labels_), 4))

*Your answer:*

## Question 10

Find a dataset where DBSCAN **loses to K-Means**.

Build two groups of different density again — 120 tight points and 120 much looser ones.
Try `eps` of 0.25, 0.45 and 0.80, and compare the best of them with `KMeans(2)`. Print the
noise count for each.

In [ ]:
rd = np.random.default_rng(5)
XD = np.vstack([rd.normal([0, 0], 0.30, (120, 2)),
                rd.normal([4.2, 0], 1.25, (120, 2))])
YD = np.r_[np.zeros(120), np.ones(120)].astype(int)

print(f"{'eps':>8}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.25, 0.45, 0.80]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XD)
    nc = len(set(d.labels_)) - (1 if -1 in d.labels_ else 0)
    print(f"{eps:>8}{nc:>10}{(d.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(YD, d.labels_):>12.4f}")

km = KMeans(2, n_init=10, random_state=0).fit(XD)
print(f"{'K-Means':>8}{2:>10}{'--':>8}{...:>12.4f}")

---
# Part D · Choosing

## Question 11

Put everything together. For each of these three datasets, run **K-Means, ward, single
linkage and DBSCAN** and print a table of agreements.

- the wine data (`WINE_Z`, k = 3)
- `make_circles(300, factor=0.45, noise=0.05, random_state=0)` (k = 2)
- the uneven-density data `XD` (k = 2)

Choose DBSCAN's `eps` with the k-distance recipe each time, so no method gets to peek at
the answers.

In [ ]:
def eps_from_kdistance(D, min_samples=5, q=0.90):
    d, _ = NearestNeighbors(n_neighbors=min_samples).fit(D).kneighbors(D)
    return float(np.quantile(np.sort(d[:, -1]), q))

JOBS = [("wine", WINE_Z, WINE_Y, 3),
        ("circles", *make_circles(300, factor=0.45, noise=0.05, random_state=0), 2),
        ("uneven density", XD, YD, 2)]

out = {}
for nm, D, y, k in JOBS:
    Zs = StandardScaler().fit_transform(D)
    out[nm] = {
        "kmeans": ...,
        "ward": ...,
        "single": ...,
        "dbscan": ...,
    }
print(pd.DataFrame(out))

## Question 12

A colleague gives you 8,000 rows of customer data with 6 numeric columns. They tell you:

> *"I ran K-Means with k=4 because that's what we always use. Here are the four segments."*

Write **three or four sentences** recommending what they should do instead, naming the
specific checks and which methods you would try. Then say which single piece of evidence
would most change your recommendation.

*Your recommendation:*


*What would most change it:*

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 6, 9 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.

Solutions will be released with Day 45.